In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04 · Feature Engineering Trust & Safety
# MAGIC
# MAGIC ## Objetivo
# MAGIC
# MAGIC Construir características reutilizables para analizar
# MAGIC comportamiento potencialmente anómalo en Reviews.
# MAGIC
# MAGIC Este notebook NO determina fraude.
# MAGIC
# MAGIC Produce variables que posteriormente consume
# MAGIC `trust_safety_evaluacion`.
# MAGIC
# MAGIC ## Salida
# MAGIC
# MAGIC `workspace.yelp_silver.slv_review_features`
# MAGIC
# MAGIC ## Grano
# MAGIC
# MAGIC **1 fila = 1 Review**


# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

import uuid


# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "yelp_silver"

SILVER = f"{CATALOG}.{SILVER_SCHEMA}"

REVIEW_TABLE = f"{SILVER}.slv_review"
USER_TABLE = f"{SILVER}.slv_user"
BUSINESS_TABLE = f"{SILVER}.slv_business"

FEATURE_TABLE = (
    f"{SILVER}.slv_review_features"
)

PIPELINE_NAME = (
    "04_feature_engineering_ts"
)

FEATURE_EXECUTION_ID = str(
    uuid.uuid4()
)


print("=" * 72)
print("YELPCONNECT - FEATURE ENGINEERING TRUST & SAFETY")
print("=" * 72)

print(
    f"Execution ID : {FEATURE_EXECUTION_ID}"
)

print(
    f"Output       : {FEATURE_TABLE}"
)

print("=" * 72)


# COMMAND ----------

# ============================================================
# 3. VALIDACIONES
# ============================================================

required_tables = [
    REVIEW_TABLE,
    USER_TABLE,
    BUSINESS_TABLE
]


for table_name in required_tables:

    if not spark.catalog.tableExists(
        table_name
    ):

        raise Exception(
            f"""
            ❌ Tabla requerida no encontrada:

            {table_name}

            Ejecutar primero Silver y DQ Silver.
            """
        )


print(
    "✔ Tablas Silver disponibles."
)


# COMMAND ----------

# ============================================================
# 4. REVIEWS
# ============================================================

reviews = (

    spark.table(
        REVIEW_TABLE
    )

    .select(

        "review_id",
        "user_id",
        "business_id",

        "stars",
        "review_text",
        "review_ts",

        "useful",
        "funny",
        "cool",

        "_silver_execution_id"
    )
)


review_count = (
    reviews.count()
)


print(
    f"Reviews Silver: {review_count:,}"
)


# COMMAND ----------

# ============================================================
# 5. USERS
# ============================================================

users = (

    spark.table(
        USER_TABLE
    )

    .select(

        "user_id",

        "yelping_since"
    )
)


# COMMAND ----------

# ============================================================
# 6. BUSINESSES
# ============================================================

businesses = (

    spark.table(
        BUSINESS_TABLE
    )

    .select(

        "business_id",

        F.col(
            "estrellas"
        ).alias(
            "business_stars"
        )
    )
)


# COMMAND ----------

# ============================================================
# 7. BASE REVIEW + USER + BUSINESS
# ============================================================

features = (

    reviews

    .join(
        users,
        "user_id",
        "left"
    )

    .join(
        businesses,
        "business_id",
        "left"
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## Features básicas
# MAGIC
# MAGIC - fecha/hora
# MAGIC - desviación del rating
# MAGIC - antigüedad de cuenta
# MAGIC - normalización del texto


# COMMAND ----------

# ============================================================
# 8. FEATURES BÁSICAS
# ============================================================

features = (

    features

    # --------------------------------------------------------
    # Timestamp como epoch
    # --------------------------------------------------------

    .withColumn(
        "review_epoch",

        F.col(
            "review_ts"
        ).cast(
            "long"
        )
    )


    # --------------------------------------------------------
    # Fecha
    # --------------------------------------------------------

    .withColumn(
        "review_date",

        F.to_date(
            "review_ts"
        )
    )


    # --------------------------------------------------------
    # Hora
    # --------------------------------------------------------

    .withColumn(
        "review_hour",

        F.hour(
            "review_ts"
        )
    )


    # --------------------------------------------------------
    # Día de semana
    # 1 = domingo
    # 7 = sábado
    # --------------------------------------------------------

    .withColumn(
        "review_day_of_week",

        F.dayofweek(
            "review_ts"
        )
    )


    # --------------------------------------------------------
    # Desviación rating Review vs Business
    # --------------------------------------------------------

    .withColumn(
        "rating_deviation",

        F.abs(
            F.col(
                "stars"
            )
            -
            F.col(
                "business_stars"
            )
        )
    )


    # --------------------------------------------------------
    # Antigüedad de cuenta al momento de Review
    # --------------------------------------------------------

    .withColumn(
        "account_age_days",

        F.datediff(

            F.to_date(
                "review_ts"
            ),

            F.to_date(
                "yelping_since"
            )
        )
    )


    # --------------------------------------------------------
    # Texto normalizado
    # --------------------------------------------------------

    .withColumn(
        "normalized_text",

        F.lower(

            F.regexp_replace(

                F.trim(
                    F.col(
                        "review_text"
                    )
                ),

                r"\s+",

                " "
            )
        )
    )


    # --------------------------------------------------------
    # Longitud texto
    # --------------------------------------------------------

    .withColumn(
        "text_length",

        F.length(
            "normalized_text"
        )
    )


    # --------------------------------------------------------
    # Hash texto
    #
    # No agrupamos textos vacíos como si fueran iguales.
    # --------------------------------------------------------

    .withColumn(
        "normalized_text_hash",

        F.when(

            F.length(
                F.col(
                    "normalized_text"
                )
            ) > 0,

            F.sha2(
                F.col(
                    "normalized_text"
                ),
                256
            )
        )
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## Actividad temporal del usuario


# COMMAND ----------

# ============================================================
# 9. WINDOWS TEMPORALES USER
# ============================================================

w_user_30m = (

    Window

    .partitionBy(
        "user_id"
    )

    .orderBy(
        F.col(
            "review_epoch"
        )
    )

    .rangeBetween(
        -30 * 60,
        0
    )
)


w_user_1h = (

    Window

    .partitionBy(
        "user_id"
    )

    .orderBy(
        F.col(
            "review_epoch"
        )
    )

    .rangeBetween(
        -60 * 60,
        0
    )
)


w_user_24h = (

    Window

    .partitionBy(
        "user_id"
    )

    .orderBy(
        F.col(
            "review_epoch"
        )
    )

    .rangeBetween(
        -24 * 60 * 60,
        0
    )
)


# COMMAND ----------

# ============================================================
# 10. FEATURES TEMPORALES USER
# ============================================================

features = (

    features

    .withColumn(
        "user_reviews_30m",

        F.count("*")
        .over(
            w_user_30m
        )
    )

    .withColumn(
        "user_reviews_1h",

        F.count("*")
        .over(
            w_user_1h
        )
    )

    .withColumn(
        "user_reviews_24h",

        F.count("*")
        .over(
            w_user_24h
        )
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## Comportamiento global User y User-Business


# COMMAND ----------

# ============================================================
# 11. WINDOWS USER
# ============================================================

w_user = (

    Window

    .partitionBy(
        "user_id"
    )
)


w_user_business = (

    Window

    .partitionBy(
        "user_id",
        "business_id"
    )
)


# COMMAND ----------

# ============================================================
# 12. FEATURES USER / BUSINESS
# ============================================================

features = (

    features

    .withColumn(
        "user_review_count",

        F.count("*")
        .over(
            w_user
        )
    )


    .withColumn(
        "user_business_review_count",

        F.count("*")
        .over(
            w_user_business
        )
    )


    .withColumn(
        "user_business_concentration",

        F.when(

            F.col(
                "user_review_count"
            ) > 0,

            F.col(
                "user_business_review_count"
            )

            /

            F.col(
                "user_review_count"
            )
        )
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## Repetición de texto


# COMMAND ----------

# ============================================================
# 13. WINDOW TEXTO
# ============================================================

w_text = (

    Window

    .partitionBy(
        "normalized_text_hash"
    )
)


# COMMAND ----------

# ============================================================
# 14. TEXT REPEAT COUNT
# ============================================================

features = (

    features

    .withColumn(
        "text_repeat_count",

        F.when(

            F.col(
                "normalized_text_hash"
            ).isNotNull(),

            F.count("*")
            .over(
                w_text
            )
        )

        .otherwise(
            F.lit(0)
        )
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## Actividad diaria del Business
# MAGIC
# MAGIC El promedio corresponde a días donde el Business
# MAGIC tuvo Reviews dentro del dataset.


# COMMAND ----------

# ============================================================
# 15. BUSINESS REVIEWS POR DÍA
# ============================================================

business_day = (

    features

    .groupBy(
        "business_id",
        "review_date"
    )

    .agg(

        F.count("*")
        .alias(
            "business_reviews_day"
        )
    )
)


# COMMAND ----------

# ============================================================
# 16. PROMEDIO DIARIO BUSINESS
# ============================================================

business_average = (

    business_day

    .groupBy(
        "business_id"
    )

    .agg(

        F.avg(
            "business_reviews_day"
        )
        .alias(
            "business_avg_daily_reviews"
        ),

        F.stddev_pop(
            "business_reviews_day"
        )
        .alias(
            "business_std_daily_reviews"
        )
    )
)


# COMMAND ----------

# ============================================================
# 17. SPIKE FEATURES
# ============================================================

business_day = (

    business_day

    .join(
        business_average,
        "business_id",
        "left"
    )

    .withColumn(
        "business_spike_ratio",

        F.when(

            F.col(
                "business_avg_daily_reviews"
            ) > 0,

            F.col(
                "business_reviews_day"
            )

            /

            F.col(
                "business_avg_daily_reviews"
            )
        )
    )


    # --------------------------------------------------------
    # Z-score adicional.
    #
    # No es usado todavía por R001-R006,
    # pero nos servirá posteriormente para ML/anomalías.
    # --------------------------------------------------------

    .withColumn(
        "business_day_zscore",

        F.when(

            F.col(
                "business_std_daily_reviews"
            ) > 0,

            (
                F.col(
                    "business_reviews_day"
                )

                -

                F.col(
                    "business_avg_daily_reviews"
                )
            )

            /

            F.col(
                "business_std_daily_reviews"
            )
        )
    )
)


# COMMAND ----------

# ============================================================
# 18. JOIN BUSINESS DAILY FEATURES
# ============================================================

features = (

    features

    .join(

        business_day,

        [
            "business_id",
            "review_date"
        ],

        "left"
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## Metadata del Feature Engineering


# COMMAND ----------

# ============================================================
# 19. METADATA
# ============================================================

features = (

    features

    .withColumn(
        "feature_execution_id",

        F.lit(
            FEATURE_EXECUTION_ID
        )
    )

    .withColumn(
        "feature_processed_at",

        F.current_timestamp()
    )

    .withColumn(
        "feature_pipeline_name",

        F.lit(
            PIPELINE_NAME
        )
    )
)


# COMMAND ----------

# ============================================================
# 20. SELECCIÓN FINAL
# ============================================================

features = (

    features

    .select(

        # ----------------------------------------------------
        # Keys
        # ----------------------------------------------------

        "review_id",
        "user_id",
        "business_id",


        # ----------------------------------------------------
        # Review
        # ----------------------------------------------------

        "stars",
        "review_text",
        "review_ts",

        "useful",
        "funny",
        "cool",


        # ----------------------------------------------------
        # Contexto
        # ----------------------------------------------------

        "business_stars",
        "yelping_since",


        # ----------------------------------------------------
        # Temporal
        # ----------------------------------------------------

        "review_date",
        "review_hour",
        "review_day_of_week",


        # ----------------------------------------------------
        # Rating
        # ----------------------------------------------------

        "rating_deviation",


        # ----------------------------------------------------
        # Account
        # ----------------------------------------------------

        "account_age_days",


        # ----------------------------------------------------
        # User velocity
        # ----------------------------------------------------

        "user_reviews_30m",
        "user_reviews_1h",
        "user_reviews_24h",
        "user_review_count",


        # ----------------------------------------------------
        # User → Business
        # ----------------------------------------------------

        "user_business_review_count",
        "user_business_concentration",


        # ----------------------------------------------------
        # Texto
        # ----------------------------------------------------

        "normalized_text_hash",
        "text_length",
        "text_repeat_count",


        # ----------------------------------------------------
        # Business activity
        # ----------------------------------------------------

        "business_reviews_day",
        "business_avg_daily_reviews",
        "business_std_daily_reviews",
        "business_spike_ratio",
        "business_day_zscore",


        # ----------------------------------------------------
        # Trazabilidad
        # ----------------------------------------------------

        "_silver_execution_id",
        "feature_execution_id",
        "feature_processed_at",
        "feature_pipeline_name"
    )
)


# COMMAND ----------

# ============================================================
# 21. VALIDAR GRANO
# ============================================================

feature_count = (
    features.count()
)


distinct_review_count = (

    features

    .select(
        "review_id"
    )

    .distinct()

    .count()
)


print(
    f"Reviews Silver     : {review_count:,}"
)

print(
    f"Features generadas : {feature_count:,}"
)

print(
    f"Review ID únicos   : {distinct_review_count:,}"
)


if (
    feature_count != review_count

    or

    distinct_review_count != review_count
):

    raise Exception(
        """
        ❌ No se cumple el grano:
        1 fila = 1 Review
        """
    )


print(
    "✔ Grano 1 Review = 1 fila validado."
)


# COMMAND ----------

# ============================================================
# 22. PERSISTENCIA
#
# Se reconstruye completa porque varias features dependen del
# comportamiento histórico global.
# ============================================================

(
    features

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        FEATURE_TABLE
    )
)


print(
    f"✔ Tabla creada: {FEATURE_TABLE}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## Evidencia


# COMMAND ----------

display(

    spark.table(
        FEATURE_TABLE
    )

    .select(

        "review_id",
        "user_id",
        "business_id",

        "rating_deviation",
        "account_age_days",

        "user_reviews_30m",
        "user_reviews_1h",
        "user_reviews_24h",

        "user_business_concentration",

        "text_repeat_count",

        "business_reviews_day",
        "business_spike_ratio"
    )

    .orderBy(
        F.desc(
            "user_reviews_30m"
        )
    )

    .limit(100)
)


# COMMAND ----------

# ============================================================
# 23. PERFIL RÁPIDO
# ============================================================

display(

    spark.table(
        FEATURE_TABLE
    )

    .agg(

        F.count("*")
        .alias(
            "reviews"
        ),

        F.max(
            "user_reviews_30m"
        )
        .alias(
            "max_reviews_30m"
        ),

        F.max(
            "user_reviews_24h"
        )
        .alias(
            "max_reviews_24h"
        ),

        F.max(
            "text_repeat_count"
        )
        .alias(
            "max_text_repeat"
        ),

        F.max(
            "business_spike_ratio"
        )
        .alias(
            "max_business_spike_ratio"
        )
    )
)


# COMMAND ----------

print()
print("=" * 72)
print("FEATURE ENGINEERING FINALIZADO")
print("=" * 72)

print(
    f"Execution ID : {FEATURE_EXECUTION_ID}"
)

print(
    f"Features     : {feature_count:,}"
)

print(
    f"Output       : {FEATURE_TABLE}"
)

print("=" * 72)